# QFabric — Overview & Run Order

**Quantum Network Emulation on FABRIC: BB84 QKD over a P4 programmable data plane.**

Alice (photon source) → BMv2 P4 switch (fiber-loss quantum channel, EtherType `0x7101`) → Bob (detector).
The classical channel rides the **same switch** as raw `0x7102` frames (TCP is a dev fallback). Slices are
single-site by default: photons cannot cross a WAN, so *distance is emulated* by one knob that drives both
fiber loss and classical propagation delay; cross-site / netem runs are explicit stress studies.

## Run the notebooks in this order

Notebooks are grouped into folders **by workflow**. The numbers still give the global order — open `00_overview` first, then follow whichever track you need.

### `fabric/` — deploy & run on a FABRIC slice
| # | Notebook | What it does | Where it runs |
|---|----------|--------------|---------------|
| 1 | `fabric/01_setup_slice` | Provision the slice, install BMv2, compile P4, start the switch | FABRIC JupyterHub |
| 2 | `fabric/02_run_experiment` | Run BB84 across the slice, collect results, verify | FABRIC JupyterHub |
| 4 | `fabric/04_analysis` | Load results and generate all plots & tables | Anywhere |
| 5 | `fabric/05_run_all_scenarios` | Run **every** scenario (singles + sweeps) with 4-way cross-validation (QFabric measured / QFabric-sim / SeQUeNCe / NetSquid) + sweep figures | FABRIC JupyterHub |
| 6 | `fabric/06_network_effects` | Classical-network (latency/jitter/loss) impact on QKD throughput | FABRIC JupyterHub |

### `sequence/` — distributed SeQUeNCe runtime
| # | Notebook | What it does | Where it runs |
|---|----------|--------------|---------------|
| 7 | `sequence/07_sequence_emulator` | Distributed **SeQUeNCe** BB84 over the P4 path | FABRIC / loopback |
| 8 | `sequence/08_sequence_scenarios` | SeQUeNCe-emulator scenario sweeps | Anywhere |
| 9 | `sequence/09_entanglement_e91` | Entanglement-based QKD (**E91 / BBM92**); CHSH Bell test | Anywhere |

### `concepts/` — QKD teaching demos (local sim, no slice)
| # | Notebook | What it does | Where it runs |
|---|----------|--------------|---------------|
| 10 | `concepts/10_eavesdropper` | Intercept-resend attack: QBER vs Eve's tap fraction, the ~11% threshold | Anywhere |
| 11 | `concepts/11_reconciliation` | Cascade reconciliation → Alice's and Bob's keys match | Anywhere |
| 12 | `concepts/12_repeater` | Entanglement swapping / repeater chains (Werner law, CHSH vs hops) | Anywhere |
| 13 | `concepts/13_qkd_security` | Finite-key bounds, authenticated channel, biased-basis, live decoy | Anywhere |

> On-slice versions of the concept demos live in `concepts/fabric/` (`*_fabric`, run locally / gitignored).

## Prerequisites
- A FABRIC account, a project, and your tokens configured in JupyterHub (for notebooks 1–2).
- `pip install -r requirements.txt` from the project root (Python 3.11 recommended).
- For the cross-validation (notebook 5): your netsquid.org credentials in `NETSQUID_USER` / `NETSQUID_PASS` (notebook 5 builds SeQUeNCe/NetSquid/QFabric-sim envs on the slice nodes via `deploy_fabric.setup_sim_envs`). See the README.

### At a glance
- **Purpose:** orient you and confirm your environment before touching a slice.
- **Inputs:** none.
- **Outputs:** an environment report (core deps, optional simulators, fablib).
- **Runs on / runtime:** anywhere; < 1 min.
- **If something fails:** if a *core* dep is MISSING, run `pip install -r requirements.txt` (Python 3.11 recommended) and re-run.

## Environment check
Confirms required deps and reports which optional simulators / fablib are present.

In [ ]:
import sys, importlib
from pathlib import Path

PROJECT_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'qne').is_dir())
sys.path.insert(0, str(PROJECT_DIR))

def check(mod, required=False):
    try:
        m = importlib.import_module(mod)
        print(f'  [OK]      {mod:24s} {getattr(m, "__version__", "")}')
        return True
    except Exception as e:
        print(f'  {"[MISSING]" if required else "[optional]"} {mod:24s} ({e.__class__.__name__})')
        return False

print(f'Python {sys.version.split()[0]}  |  project: {PROJECT_DIR}\n')
print('Core (required):')
core_ok = all([check('numpy', True), check('yaml', True), check('qne', True), check('validation', True)])
print('\nOptional cross-validation backends (notebook 5):')
check('sequence'); check('netsquid')
print('\nFABRIC deployment (notebooks 1-2):')
check('fabrictestbed_extensions')
print('\nCore environment OK — continue to 01_setup_slice' if core_ok else '\nInstall core deps: pip install -e .')

## Quick smoke test (optional, runs anywhere)
Runs the pure-Python cross-validation on the baseline scenario. With no simulators installed it reports QFabric only and marks the others **SKIPPED** (never a false pass).

In [ ]:
import subprocess
out = subprocess.run([sys.executable, '-m', 'validation.compare',
                      'validation/scenarios/baseline_1km.yml'],
                     cwd=str(PROJECT_DIR), capture_output=True, text=True)
print(out.stdout or out.stderr)

---
**Next:** `01_setup_slice` to provision the FABRIC slice.